In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import random
from torch.optim import LBFGS, Adam
from tqdm import tqdm
import optuna
from model_components.models import DecoderOnlyPINNsformer, PINNs
from model_components.util import *
seed = 0
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
res, b_left, b_right, b_upper, b_lower = get_data([0,1], [0,1], 101, 101)
res_test, _, _, _, _ = get_data([0,1], [0,1], 101, 101)

def u_ana(x,t):
    return np.sin(np.pi*x) * np.cos(2*np.pi*t) + 0.5 * np.sin(3*np.pi*x) * np.cos(6*np.pi*t)

u_true = u_ana(res_test[:,0], res_test[:,1]).reshape(101,101)
    
res_test = torch.tensor(res_test, dtype=torch.float32, requires_grad=True).to(device)
x_test, t_test = res_test[:,0:1], res_test[:,1:2]   

def set_seed(seed=0):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)



res = torch.tensor(res, dtype=torch.float32, requires_grad=True).to(device)
b_left = torch.tensor(b_left, dtype=torch.float32, requires_grad=True).to(device)
b_right = torch.tensor(b_right, dtype=torch.float32, requires_grad=True).to(device)
b_upper = torch.tensor(b_upper, dtype=torch.float32, requires_grad=True).to(device)
b_lower = torch.tensor(b_lower, dtype=torch.float32, requires_grad=True).to(device)

x_res, t_res = res[:,0:1], res[:,1:2]
x_left, t_left = b_left[:,0:1], b_left[:,1:2]
x_right, t_right = b_right[:,0:1], b_right[:,1:2]
x_upper, t_upper = b_upper[:,0:1], b_upper[:,1:2]
x_lower, t_lower = b_lower[:,0:1], b_lower[:,1:2]

smallest_rl1 = 1e10

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform(m.weight)
        m.bias.data.fill_(0.01)



set_seed(0)

d_hidden = 768
num_layer = 6

model = PINNs(in_dim=2, hidden_dim=d_hidden, out_dim=1, num_layer=num_layer).to(device)

model.apply(init_weights)
optim = LBFGS(model.parameters(), line_search_fn='strong_wolfe')

model.load_state_dict(torch.load('saves/1dw_pinn_trial_34.pth', map_location=device))
# optim = Adam(model.parameters(), lr=1e-4)

n_params = get_n_params(model)




with torch.no_grad():
    pred = model(x_test, t_test)[:,0:1]
    pred = pred.cpu().detach().numpy()

pred = pred.reshape(101,101)
rl1 = np.sum(np.abs(u_true-pred)) / np.sum(np.abs(u_true))
rl2 = np.sqrt(np.sum((u_true-pred)**2) / np.sum(u_true**2))
print(f'RL1: {rl1}, RL2: {rl2}, N params: {n_params}')

/home/ubuntu/pinns/pinns/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_34540/2354866908.py:53: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(m.weight)


RL1: 0.022546166630196453, RL2: 0.02316848820302941, N params: 2365441
